# Manage binance_bitcoin
Edit the YAML below and run this notebook in Databricks to create or replace the job settings. This deploys the definition; it does not run the ETL.

Commit this notebook in your Databricks Git folder to version the YAML. A commit or merge alone does not deploy it. Run this notebook again after pulling the desired version.

The YAML is the complete desired job configuration: settings and tasks omitted from it are removed/reset on deployment. Existing job ID and run history are retained. Do not run concurrent deployments. This is a small Jobs API deployer, not a bundle engine: bundle variables, permissions and other bundle-only directives are not resolved.

Uses serverless job configuration for Free Edition. Your workspace must support the environment version and performance target specified below. Install dependencies in the notebook environment if `%pip` is unavailable.


In [0]:
%%capture --no-stderr
%pip install -q -U databricks-sdk pyyaml



In [0]:
dbutils.library.restartPython()

## Edit the job
Set JOB_ID to an existing job ID to target it explicitly, especially before renaming. With None, the notebook finds an exact name match, stops if ambiguous, or creates a job if none exists. After first deployment, copy the printed job ID here for stable targeting.

In [0]:
from pathlib import Path
from project_config import config
databricks_project_root = config["databricks_project_root"]

JOB_ID = 1028404418479387

JOB_YAML = Path(
    f"{databricks_project_root}/orchestration/jobs/bitcoin_data_project.yml"
).read_text(encoding="utf-8")

## Apply the definition
Uses notebook-native authentication; no token is embedded in the notebook. You need permission to create or manage the job.

In [0]:
import yaml
from databricks.sdk import WorkspaceClient

settings = yaml.safe_load(JOB_YAML)["resources"]["jobs"]["binance_bitcoin"]
if not settings.get("name") or not settings.get("tasks"):
    raise ValueError("The job must have a name and at least one task.")

w = WorkspaceClient()
job_id = JOB_ID

if job_id is None:
    matches = [job for job in w.jobs.list(name=settings["name"])
               if job.settings and job.settings.name == settings["name"]]
    if len(matches) > 1:
        raise ValueError("Multiple jobs have this name. Set JOB_ID explicitly.")
    if matches:
        job_id = matches[0].job_id

if job_id is None:
    result = w.api_client.do("POST", "/api/2.2/jobs/create", body=settings)
    job_id = result["job_id"]
    action = "Created"
else:
    w.api_client.do("POST", "/api/2.2/jobs/reset",
                    body={"job_id": job_id, "new_settings": settings})
    action = "Updated"

print(f"{action}: {settings['name']} | job_id={job_id}")
print(f"{w.config.host.rstrip('/')}/#job/{job_id}")


## Versioning and rollback
Commit the notebook (or the separate YAML) in Git. To restore an earlier definition, restore that Git version and run the deployment cell against the same JOB_ID. This restores job configuration, not data produced by previous runs.

For automatic deployment on merge to main, use a GitHub Actions workflow with Databricks authentication. Databricks bundles are the native declarative option and can also be deployed manually without CI/CD.

References:
- https://docs.databricks.com/api/jobs/v2/job
- https://docs.databricks.com/aws/en/dev-tools/sdk-python
- https://docs.databricks.com/aws/en/dev-tools/bundles/jobs-tutorial

Validated locally for Python syntax and notebook structure; not executed against your workspace.
